# Module 3: Light-Matter Interaction & Cavity QED
## Notebook 3.7: Electromagnetically Induced Transparency (EIT) & Dark States

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_3_light_matter_cavity_qed/08_electromagnetically_induced_transparency_eit.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** **Light-Matter Interaction & Cavity QED**
  * **3.1** Two-Level Atoms & Optical Bloch Equations
  * **3.2** Rabi Oscillations, Detuning & Pulse Control
  * **3.3** The Jaynes-Cummings Hamiltonian & RWA
  * **3.4** The Quantum Rabi Model & Dressed States
  * **3.5** Quantum Collapse and Revival of Atomic Inversion
  * **3.6** Resonance Fluorescence & The Mollow Triplet
  * **3.7 Electromagnetically Induced Transparency (EIT) & Dark States** $\leftarrow$ *(You are here)*

---

### Learning Objectives
1. **The Three-Level $\Lambda$-System:** Formulate the Hamiltonian and Lindblad master equation for two stable ground states coupled to a common radiative excited state.
2. **Coherent Population Trapping (CPT) & Dark States:** Derive the uncoupled dark eigenstate $|D\rangle = \cos\theta |1\rangle - \sin\theta |2\rangle$ and prove that it has identically zero probability of occupying the dissipative excited state $|3\rangle$.
3. **Electromagnetically Induced Transparency (EIT):** Calculate the complex probe susceptibility $\chi(\Delta_p)$, demonstrate the cancellation of resonant absorption via quantum interference, and evaluate the transparency window linewidth.
4. **Steep Dispersion & Slow/Stopped Light:** Connect the steep normal dispersion $dn/d\omega > 0$ to orders-of-magnitude reductions in group velocity $v_g \ll c$ and explain the storage of optical quantum information into atomic spin waves.
5. **STIRAP (Stimulated Raman Adiabatic Passage):** Model the counter-intuitive pulse sequence and numerically demonstrate $100\%$ population transfer $|1\rangle \to |2\rangle$ with zero excited state fluorescence.


In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

try:
    import qutip as qt
    HAS_QUTIP = True
    print("QuTiP successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


---

## 1. The Three-Level $\Lambda$-System & Dark States

Consider an atom with two stable (or metastable) lower energy states $|1\rangle$ and $|2\rangle$, coupled to a common radiative excited state $|3\rangle$ in a $\Lambda$-configuration:
* A weak **probe laser** with Rabi frequency $\Omega_p$ and detuning $\Delta_p = \omega_p - \omega_{31}$ drives transition $|1\rangle \leftrightarrow |3\rangle$.
* A strong **control (coupling) laser** with Rabi frequency $\Omega_c$ and detuning $\Delta_c = \omega_c - \omega_{32}$ drives transition $|2\rangle \leftrightarrow |3\rangle$.

In the rotating frame, the semiclassical interaction Hamiltonian under the RWA is:
$$\hat{H} = -\hbar \Delta_p |1\rangle\langle 1| - \hbar (\Delta_p - \Delta_c) |2\rangle\langle 2| + \frac{\hbar}{2} \begin{pmatrix} 0 & 0 & \Omega_p \\ 0 & 0 & \Omega_c \\ \Omega_p & \Omega_c & 0 \end{pmatrix}$$

At **two-photon Raman resonance** ($\delta = \Delta_p - \Delta_c = 0$), setting on-resonance detunings $\Delta_p = \Delta_c = 0$, the Hamiltonian in basis $\{|1\rangle, |2\rangle, |3\rangle\}$ is:
$$\hat{H}_0 = \frac{\hbar}{2} \begin{pmatrix} 0 & 0 & \Omega_p \\ 0 & 0 & \Omega_c \\ \Omega_p & \Omega_c & 0 \end{pmatrix}$$

### The Dark State and Bright State
Diagonalizing $\hat{H}_0$ yields three eigenstates:
1. **The Dark State $|D\rangle$ (Zero Eigenvalue $E_D = 0$):**
   $$|D\rangle = \frac{\Omega_c |1\rangle - \Omega_p |2\rangle}{\sqrt{\Omega_c^2 + \Omega_p^2}} = \cos\theta |1\rangle - \sin\theta |2\rangle$$
   where $\tan\theta = \frac{\Omega_p}{\Omega_c}$.
2. **The Bright States $|B_\pm\rangle$ (Eigenvalues $E_\pm = \pm \frac{\hbar}{2}\sqrt{\Omega_p^2 + \Omega_c^2}$):**
   $$|B_\pm\rangle = \frac{1}{\sqrt{2}} \left( \sin\theta |1\rangle + \cos\theta |2\rangle \pm |3\rangle \right)$$

### The Coherent Population Trapping (CPT) Principle:
Notice that the dark state $|D\rangle$ is a coherent superposition of **ground states only** with **identically zero projection onto the excited state $|3\rangle$**:
$$\langle 3 | D \rangle \equiv 0$$
Because $|D\rangle$ has zero overlap with the lossy state $|3\rangle$:
$$\hat{H}_0 |D\rangle = 0 \implies \mathcal{D}[\hat{L}_{\text{decay}}]|D\rangle = 0$$
An atom prepared in $|D\rangle$ **cannot absorb light, cannot be excited, and cannot emit spontaneous fluorescence**! The atom is trapped in a non-absorbing dark superposition.

---

## 2. Electromagnetically Induced Transparency (EIT)

In the absence of the control laser ($\Omega_c = 0$), a resonant probe laser ($\Delta_p = 0$) encounters strong Beer-Lambert absorption from the $|1\rangle \to |3\rangle$ atomic transition: the medium is completely opaque.

When a strong control laser ($\Omega_c \gg \Gamma_3$) is illuminated on the adjacent transition $|2\rangle \leftrightarrow |3\rangle$, the probe field experiences two competing quantum absorption paths:
1. Direct transition: $|1\rangle \to |3\rangle$
2. Indirect transition: $|1\rangle \to |3\rangle \to |2\rangle \to |3\rangle$

The transition amplitudes interfere **destructively**. Solving the steady-state Lindblad master equation for the probe susceptibility $\chi(\Delta_p) = \chi'(\Delta_p) + i\chi''(\Delta_p)$ in the weak probe limit ($\Omega_p \ll \Omega_c$):
$$\chi(\Delta_p) \propto \frac{\Delta_p + i\gamma_{12}}{(\Delta_p + i\gamma_{12})(\Delta_p + i\Gamma_3/2) - |\Omega_c/2|^2}$$
where $\Gamma_3$ is the excited state radiative decay rate, and $\gamma_{12}$ is the ground-state dephasing rate.

In the ideal limit ($\gamma_{12} \to 0$):
$$\chi''(\Delta_p = 0) = 0 \implies \text{Absorption drops to exactly ZERO!}$$
The opaque atomic cloud becomes **completely transparent** over an EIT window of width:
$$\Delta\omega_{\text{EIT}} \approx \frac{|\Omega_c|^2}{\Gamma_3}$$

---

## 3. Slow Light, Stopped Light & STIRAP

### 1. Ultra-Slow Group Velocity
The real part of the refractive index $n(\omega) \approx 1 + \frac{1}{2}\chi'(\omega)$ undergoes extraordinarily steep **normal dispersion** ($dn/d\omega > 0$) inside the narrow transparency window.
The group velocity of a probe pulse propagating through the EIT medium is:
$$v_g = \frac{c}{n + \omega \frac{dn}{d\omega}} \approx \frac{c}{\omega \frac{dn}{d\omega}} \approx \frac{2\hbar \epsilon_0 c}{\mathcal{N} d_{13}^2 \omega_{31}} \frac{|\Omega_c|^2}{\Gamma_3} \ll c$$
In atomic vapors, group velocities as slow as $17\text{ m/s}$ (bicycle speed!) or slower have been demonstrated.

### 2. Quantum Memory / Stopped Light:
If the control laser is adiabatically turned off ($\Omega_c(t) \to 0$) while the slowed probe pulse is spatially compressed inside the atomic vapor, the photonic excitation is completely converted into a long-lived ground-state **atomic spin coherence (dark-state polariton)**. Ramping $\Omega_c$ back up recreates the traveling optical pulse, realizing a **coherent quantum optical memory**!

### 3. STIRAP: Stimulated Raman Adiabatic Passage
To transfer atomic population from $|1\rangle$ to $|2\rangle$ with $100\%$ efficiency, one applies a **counter-intuitive pulse sequence**:
1. The **control pulse $\Omega_c(t)$ arrives FIRST**, establishing $\tan\theta = \Omega_p / \Omega_c \to 0 \implies |D\rangle \approx |1\rangle$.
2. The atom starts in $|1\rangle$, so it is automatically in the dark state!
3. The **probe pulse $\Omega_p(t)$ arrives LATER** as $\Omega_c(t)$ is smoothly turned off.
4. The mixing angle rotates adiabatically from $\theta(t_i) = 0 \to \theta(t_f) = \pi/2$:
   $$|D(t_f)\rangle = \cos(\pi/2)|1\rangle - \sin(\pi/2)|2\rangle = -|2\rangle$$
All population is transferred to $|2\rangle$ with **zero population ever entering the dissipative excited state $|3\rangle$**!


In [ ]:
# Implementation: EIT Complex Susceptibility and STIRAP Dynamic Solvers

class EITSimulator:
    """
    Simulates Electromagnetically Induced Transparency (EIT) complex susceptibility,
    slow light group velocity, and STIRAP adiabatic state transfer.
    """
    def __init__(self, omega_c=3.0, gamma3=1.0, gamma12=0.01):
        self.omega_c = float(omega_c)
        self.gamma3 = float(gamma3)
        self.gamma12 = float(gamma12)
        
    def susceptibility(self, delta_p_arr):
        """
        Calculates complex probe susceptibility chi(Delta_p) = chi' + i*chi''
        in the linear probe regime (Omega_p << Omega_c).
        """
        # Complex detunings
        d_p = delta_p_arr
        term1 = d_p + 1j * self.gamma12
        term2 = d_p + 1j * (0.5 * self.gamma3)
        
        # chi \propto term1 / (term1 * term2 - (Omega_c / 2)^2)
        denom = term1 * term2 - (0.5 * self.omega_c)**2
        chi = 1j * (self.gamma3 / 2.0) * (term1 / denom)
        return chi
        
    def group_velocity_ratio(self, omega_c_arr, N_density_scale=1.0):
        """
        Calculates group velocity ratio v_g / c at transparency center (Delta_p = 0):
        v_g / c \approx (Omega_c^2) / (Omega_c^2 + N_scale * Gamma3)
        """
        return omega_c_arr**2 / (omega_c_arr**2 + N_density_scale * self.gamma3 * 100.0)
        
    def simulate_stirap(self, t_span=(-15.0, 15.0), t_delay=4.0, t_width=3.5, n_points=1000):
        """
        Simulates the counter-intuitive STIRAP pulse sequence:
        Omega_c(t) peaks first at -t_delay/2, Omega_p(t) peaks second at +t_delay/2.
        """
        t_arr = np.linspace(t_span[0], t_span[1], n_points)
        
        # Gaussian pulse envelopes
        om_c_t = 5.0 * np.exp(-((t_arr + 0.5 * t_delay) / t_width)**2)
        om_p_t = 5.0 * np.exp(-((t_arr - 0.5 * t_delay) / t_width)**2)
        
        # State vector |psi(t)> = [c1, c2, c3]
        def stirap_odes(t, y):
            c1, c2, c3 = y[0] + 1j*y[1], y[2] + 1j*y[3], y[4] + 1j*y[5]
            
            # Pulse values at time t
            oc = 5.0 * np.exp(-((t + 0.5 * t_delay) / t_width)**2)
            op = 5.0 * np.exp(-((t - 0.5 * t_delay) / t_width)**2)
            
            # H = hbar/2 * [[0, 0, op], [0, 0, oc], [op, oc, 0]]
            # Decay: state 3 decays with rate gamma3/2
            dc1 = -0.5j * op * c3
            dc2 = -0.5j * oc * c3
            dc3 = -0.5j * (op * c1 + oc * c2) - 0.5 * self.gamma3 * c3
            
            return [dc1.real, dc1.imag, dc2.real, dc2.imag, dc3.real, dc3.imag]
            
        # Initial condition: all atoms in ground state |1>
        y0 = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        sol = solve_ivp(stirap_odes, t_span, y0, t_eval=t_arr, method='RK45', rtol=1e-8, atol=1e-10)
        
        p1 = sol.y[0]**2 + sol.y[1]**2
        p2 = sol.y[2]**2 + sol.y[3]**2
        p3 = sol.y[4]**2 + sol.y[5]**2
        
        return t_arr, om_c_t, om_p_t, p1, p2, p3

# Verification Benchmark: Confirm Zero On-Resonance Absorption in Ideal EIT Limit
sim_test = EITSimulator(omega_c=4.0, gamma3=1.0, gamma12=0.0)
chi_res = sim_test.susceptibility(np.array([0.0]))[0]
abs_res = np.imag(chi_res)

# STIRAP test: Confirm >94% population transfer with <2% max excited state population
t_test, _, _, p1_res, p2_res, p3_res = sim_test.simulate_stirap()
final_p2 = p2_res[-1]
max_p3 = np.max(p3_res)

print("--- EIT & STIRAP Benchmark Verification ---")
print(f"On-Resonance Absorption Im(chi) (gamma12=0): {abs_res:.6e} (Expected strictly 0.0)")
print(f"STIRAP Final Target Population P2(t_end):    {final_p2*100:.2f}% (Target > 94.0%)")
print(f"STIRAP Peak Excited Population P3_max:       {max_p3*100:.4f}% (Target < 2.0%)")

assert abs(abs_res) < 1e-12, "EIT transparency zero absorption check failed!"
assert final_p2 > 0.94, "STIRAP population transfer check failed!"
assert max_p3 < 0.02, "STIRAP dark state excited population suppression check failed!"
print("Verification Passed: Perfect EIT cancellation and dark-state adiabatic transfer verified!")


In [ ]:
# Publication-Quality 4-Panel Visualization: EIT Spectrum, Dispersion, Slow Light & STIRAP Transfer

fig = plt.figure(figsize=(16, 12))

# =============================================================================
# Panel 1: EIT Probe Absorption Profile Im(chi) vs. Control Field Strength
# =============================================================================
ax1 = fig.add_subplot(2, 2, 1)

delta_scan = np.linspace(-5.0, 5.0, 1000)
ctrl_powers = [0.0, 1.0, 2.5, 5.0]
colors = ['gray', '#1f77b4', '#2ca02c', '#d62728']

for oc, col in zip(ctrl_powers, colors):
    sim_oc = EITSimulator(omega_c=oc, gamma3=1.0, gamma12=0.02)
    chi_vals = sim_oc.susceptibility(delta_scan)
    absorption = np.imag(chi_vals)
    label_txt = r"Two-Level Lorentzian ($\Omega_c = 0$)" if oc == 0 else rf"$\Omega_c = {oc:.1f}\Gamma$"
    style = '--' if oc == 0 else '-'
    ax1.plot(delta_scan, absorption, color=col, linestyle=style, lw=2.0, label=label_txt)

ax1.set_xlabel(r"Probe Detuning $\Delta_p / \Gamma_3$")
ax1.set_ylabel(r"Probe Absorption $\mathrm{Im}[\chi(\Delta_p)]$")
ax1.set_title(r"Electromagnetically Induced Transparency Window")
ax1.set_ylim(-0.02, 1.08)
ax1.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 2: Kramers-Kronig Pair: Real Dispersion & Steep Normal Gradient
# =============================================================================
ax2 = fig.add_subplot(2, 2, 2)

sim_disp = EITSimulator(omega_c=2.5, gamma3=1.0, gamma12=0.005)
chi_disp = sim_disp.susceptibility(delta_scan)
re_chi = np.real(chi_disp)
im_chi = np.imag(chi_disp)

ax2.plot(delta_scan, im_chi, 'r-', lw=2.2, label=r"Absorption $\mathrm{Im}(\chi)$ (Transparency Dip)")
ax2.plot(delta_scan, re_chi, 'b--', lw=2.2, label=r"Dispersion $\mathrm{Re}(\chi)$ (Steep Normal Slope)")
ax2.axvline(0.0, color='k', linestyle=':', alpha=0.5)

# Annotate steep gradient
ax2.annotate(r"Steep Normal Dispersion $\frac{dn}{d\omega} > 0$", xy=(0.0, 0.0), xytext=(0.8, -0.45),
             arrowprops=dict(arrowstyle='->', color='navy', lw=1.5), fontsize=10, color='navy')

ax2.set_xlabel(r"Probe Detuning $\Delta_p / \Gamma_3$")
ax2.set_ylabel("Susceptibility Component")
ax2.set_title(r"Kramers-Kronig Relations: Ultra-Steep Dispersion ($v_g \ll c$)")
ax2.set_ylim(-0.8, 1.05)
ax2.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 3: Deceleration of Light: Group Velocity vs. Control Rabi Frequency
# =============================================================================
ax3 = fig.add_subplot(2, 2, 3)

omega_c_scan = np.linspace(0.1, 10.0, 300)
vg_c = sim_disp.group_velocity_ratio(omega_c_scan, N_density_scale=2.0)

ax3.plot(omega_c_scan, vg_c, 'purple', lw=2.5, label=r"Normalized Group Velocity $v_g / c$")
ax3.axhline(1.0, color='gray', linestyle=':', label="Vacuum Speed of Light $c$")

# Annotate slow light and stopped light
ax3.annotate("Stopped Light (Quantum Memory)\n" r"$v_g \to 0$ as $\Omega_c \to 0$",
             xy=(0.5, 0.02), xytext=(2.0, 0.25),
             arrowprops=dict(arrowstyle='->', color='purple', lw=1.5), fontsize=9.5)

ax3.set_xlabel(r"Control Field Rabi Frequency $\Omega_c / \Gamma_3$")
ax3.set_ylabel(r"Relative Group Velocity $v_g / c$")
ax3.set_title(r"Slow Light Tuning: Dynamic Deceleration & Freezing")
ax3.set_ylim(-0.02, 1.05)
ax3.legend(loc="lower right", frameon=True)

# =============================================================================
# Panel 4: STIRAP Adiabatic Population Transfer
# =============================================================================
ax4 = fig.add_subplot(2, 2, 4)

t_s, oc_t, op_t, p1_s, p2_s, p3_s = sim_disp.simulate_stirap()

# Plot pulse envelopes (scaled)
ax4.plot(t_s, oc_t / 5.0, 'g--', lw=1.5, alpha=0.6, label=r"Control Pulse $\Omega_c(t)$ (Arrives First)")
ax4.plot(t_s, op_t / 5.0, 'orange', lw=1.5, alpha=0.6, label=r"Pump/Probe Pulse $\Omega_p(t)$ (Arrives Second)")

# Plot populations
ax4.plot(t_s, p1_s, 'b-', lw=2.2, label=r"Initial State $P_1(t)$")
ax4.plot(t_s, p2_s, 'r-', lw=2.5, label=r"Target State $P_2(t) \to 1.0$")
ax4.plot(t_s, p3_s * 50.0, 'k:', lw=1.8, label=r"Excited State $P_3(t) \times 50$ ($\approx 0$)")

ax4.set_xlabel("Time ($t / \tau$)")
ax4.set_ylabel("State Population / Pulse Amplitude")
ax4.set_title(r"STIRAP: 100% Transfer via Dark State with Zero Loss")
ax4.set_ylim(-0.05, 1.15)
ax4.legend(loc="center right", fontsize=8.5, frameon=True)

plt.tight_layout()
plt.show()


> **Physical Insight: Why Must the Control Pulse Arrive BEFORE the Pump Pulse in STIRAP?**
>
> In classical intuition, to move something from room $1$ to room $2$ through door $3$, one would first open door $1 \to 3$ (pump pulse), and then open door $3 \to 2$ (Stokes/control pulse).
>
> In quantum mechanics, following this "intuitive" sequence ($\Omega_p$ first) causes **instant disaster**: the pump laser excites atoms into state $|3\rangle$, which spontaneously radiates in random directions at rate $\Gamma_3$, causing severe irreversible decoherence and photon loss.
>
> In the **counter-intuitive STIRAP sequence** ($\Omega_c$ first):
> 1. When $\Omega_c$ is active and $\Omega_p = 0$, the dark state is:
>    $$|D\rangle = \frac{\Omega_c |1\rangle - \Omega_p |2\rangle}{\sqrt{\Omega_c^2 + \Omega_p^2}} \to |1\rangle$$
>    The atom, initially in $|1\rangle$, is **already sitting in the dark state**!
> 2. As $\Omega_p(t)$ turns on and $\Omega_c(t)$ turns off, the ratio $\tan\theta = \Omega_p/\Omega_c$ continuously sweeps from $0$ to $\infty$.
> 3. The state vector remains locked inside $|D(t)\rangle$ by the **adiabatic theorem**, rotating smoothly into:
>    $$|D\rangle \to -|2\rangle$$
> At no point in time does the state ever acquire population in the lossy excited state $|3\rangle$! This topological protection makes STIRAP one of the most robust state-engineering tools in quantum physics and atomic clocks.


In [ ]:
# Interactive Exploration: EIT Window with Ground State Dephasing

def plot_interactive_eit(omega_c=3.0, gamma12=0.01):
    """
    Simulates the EIT susceptibility and demonstrates how ground-state dephasing
    gamma_12 fills in the transparency window.
    """
    sim = EITSimulator(omega_c=omega_c, gamma3=1.0, gamma12=gamma12)
    
    delta_pts = np.linspace(-4.0, 4.0, 600)
    chi = sim.susceptibility(delta_pts)
    abs_profile = np.imag(chi)
    disp_profile = np.real(chi)
    
    fig, (ax_abs, ax_dsp) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Absorption
    ax_abs.plot(delta_pts, abs_profile, 'r-', lw=2.2, label=r"Absorption $\mathrm{Im}(\chi)$")
    ax_abs.axhline(0.0, color='gray', linestyle=':')
    ax_abs.set_xlabel(r"Probe Detuning $\Delta_p / \Gamma_3$")
    ax_abs.set_ylabel(r"Absorption $\mathrm{Im}(\chi)$")
    ax_abs.set_title(rf"EIT Transparency Dip ($\Omega_c={omega_c:.1f}\Gamma_3$, $\gamma_{{12}}={gamma12:.3f}\Gamma_3$)")
    ax_abs.set_ylim(-0.02, 1.05)
    ax_abs.legend(frameon=True)
    
    # Dispersion
    ax_dsp.plot(delta_pts, disp_profile, 'b-', lw=2.2, label=r"Dispersion $\mathrm{Re}(\chi)$")
    ax_dsp.axhline(0.0, color='gray', linestyle=':')
    ax_dsp.set_xlabel(r"Probe Detuning $\Delta_p / \Gamma_3$")
    ax_dsp.set_ylabel(r"Dispersion $\mathrm{Re}(\chi)$")
    ax_dsp.set_title("Steep Normal Dispersion Profile")
    ax_dsp.set_ylim(-0.8, 0.8)
    ax_dsp.legend(frameon=True)
    
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    widgets.interact(
        plot_interactive_eit,
        omega_c=widgets.FloatSlider(value=3.0, min=0.5, max=8.0, step=0.5, description=r"$\Omega_c/\Gamma_3$:"),
        gamma12=widgets.FloatSlider(value=0.01, min=0.0, max=0.1, step=0.01, description=r"$\gamma_{12}/\Gamma_3$:")
    )
else:
    print("Static exploration demonstration (default parameters: Omega_c = 3.0, gamma_12 = 0.01):")
    plot_interactive_eit(omega_c=3.0, gamma12=0.01)


---

## 4. Self-Check Exercises & Guided Challenge

### Challenge 1: The Dark State at Equal Rabi Frequencies
Calculate the dark state $|D\rangle$ when both probe and control lasers drive their respective transitions with equal Rabi frequencies ($\Omega_p = \Omega_c$).
Verify that the excited state population is zero, and determine the relative phase between ground states $|1\rangle$ and $|2\rangle$.

<details>
<summary><b>Solution & Derivation</b> (Click to expand)</summary>

The general dark state formula is:
$$|D\rangle = \frac{\Omega_c |1\rangle - \Omega_p |2\rangle}{\sqrt{\Omega_c^2 + \Omega_p^2}}$$
Setting $\Omega_p = \Omega_c = \Omega$:
$$|D\rangle = \frac{\Omega |1\rangle - \Omega |2\rangle}{\sqrt{\Omega^2 + \Omega^2}} = \frac{\Omega}{\sqrt{2}\Omega} (|1\rangle - |2\rangle) = \frac{1}{\sqrt{2}} (|1\rangle - |2\rangle)$$
1. The excited state component is $\langle 3|D\rangle = 0$, so $P_3 = |\langle 3|D\rangle|^2 = 0$ identically.
2. The state is an **equal superposition** with a relative phase of $\pi$ ($e^{i\pi} = -1$) between states $|1\rangle$ and $|2\rangle$.
</details>

---

### Challenge 2: Numerical Verification of STIRAP Fidelity
Simulate the counter-intuitive STIRAP sequence with pulse separation $t_{\text{delay}} = 4.0\tau$ and pulse width $\tau = 3.5$. Verify numerically that:
1. The final transfer fidelity $\mathcal{F} = P_2(t_{\text{final}}) > 94.0\%$.
2. The maximum transient population in the lossy state $P_3^{\text{max}} < 2.0\%$.


In [ ]:
# Numerical Verification of Challenge 2: STIRAP Transfer Fidelity & Dark-State Invariance

sim_c2 = EITSimulator(omega_c=5.0, gamma3=1.0, gamma12=0.0)

t_eval, oc_eval, op_eval, p1_eval, p2_eval, p3_eval = sim_c2.simulate_stirap(
    t_span=(-15.0, 15.0), t_delay=4.0, t_width=3.5, n_points=1200
)

final_transfer_fidelity = p2_eval[-1]
max_excited_population = np.max(p3_eval)

print(f"--- STIRAP High-Fidelity State Transfer Benchmark ---")
print(f"Initial State Population P1(t_start):  {p1_eval[0]:.6f}")
print(f"Final Target Population P2(t_final):   {final_transfer_fidelity:.6f} (Target > 0.940)")
print(f"Peak Intermediate Population P3_max:   {max_excited_population:.6f} (Target < 0.020)")
print(f"Transfer Efficiency:                   {final_transfer_fidelity * 100:.3f}%")

assert final_transfer_fidelity > 0.94, "STIRAP transfer fidelity benchmark failed!"
assert max_excited_population < 0.02, "STIRAP excited state suppression benchmark failed!"
print("Verification Passed: STIRAP achieves >94% state transfer with <2% transient excited population!")


---

## 5. Summary & Key Formulas

| Concept | Mathematical Expression | Physical Interpretation |
| :--- | :--- | :--- |
| **Dark State** | $|D\rangle = \cos\theta |1\rangle - \sin\theta |2\rangle$ ($\tan\theta = \frac{\Omega_p}{\Omega_c}$) | Uncoupled superposition; $\langle 3|D\rangle \equiv 0$ prevents absorption |
| **EIT Window Width** | $\Delta\omega_{\text{EIT}} \approx \frac{|\Omega_c|^2}{\Gamma_3}$ | Spectral width of transparency dip |
| **Group Velocity** | $v_g \approx \frac{c}{\omega \frac{dn}{d\omega}} \propto \frac{|\Omega_c|^2}{\Gamma_3}$ | Deceleration of light pulses via steep normal dispersion |
| **Stopped Light** | $\Omega_c(t) \to 0$ | Conversion of optical pulse into a stationary atomic spin wave |
| **STIRAP Sequence** | $\Omega_c(t)$ before $\Omega_p(t)$ | Adiabatic rotation of $|D\rangle$ from $|1\rangle$ to $|2\rangle$ with zero loss |

---

### Further Reading & Foundational Literature
* **S. E. Harris**, *Electromagnetically Induced Transparency*, Physics Today 50, 7, 36 (1997).
* **L. V. Hau, S. E. Harris, Z. Dutton, & C. H. Behroozi**, *Light Speed Reduction to 17 Metres per Second in an Ultracold Atomic Gas*, Nature 397, 594–598 (1999).
* **K. Bergmann, H. Theuer, & B. W. Shore**, *Coherent Population Transfer Among Quantum States of Atoms and Molecules*, Rev. Mod. Phys. 70, 1003 (1998).
* **M. Fleischhauer, A. Imamoglu, & J. P. Marangos**, *Electromagnetically Induced Transparency: Optics in Coherent Media*, Rev. Mod. Phys. 77, 633 (2005).

---

### Congratulations on Completing Module 3!
You have mastered the physics of light-matter interaction from semiclassical Bloch equations to fully quantized Cavity QED and coherent multi-level interference:
* **Notebook 3.1:** Two-Level Atoms & Optical Bloch Equations
* **Notebook 3.2:** Rabi Oscillations, Detuning & Pulse Control
* **Notebook 3.3:** The Jaynes-Cummings Hamiltonian & RWA
* **Notebook 3.4:** The Quantum Rabi Model & Dressed States
* **Notebook 3.5:** Quantum Collapse and Revival of Atomic Inversion
* **Notebook 3.6:** Resonance Fluorescence & The Mollow Triplet
* **Notebook 3.7:** Electromagnetically Induced Transparency (EIT) & Dark States

In **Module 4: Open Quantum Systems & Dissipation**, we advance to the full machinery of master equations, quantum jumps, Monte Carlo wavefunctions, input-output theory, and non-Hermitian photonics!
